# 🐍 Python Dictionaries: The Definitive Interview Guide & Reference

This notebook provides an exhaustive, production-grade reference and interview guide for Python dictionaries (`dict`). It is structured systematically to cover fundamental characteristics, under-the-hood CPython implementations, time and space complexities, all 11 built-in methods, hashability rules, edge cases, and high-frequency interview patterns.

---

## 🎯 1. Definition and Fundamental Characteristics

A Python dictionary is an **ordered**, **mutable**, **dynamic**, and **associative (key-value)** collection of unique, hashable keys mapped to arbitrary values.

### 🔑 Core Properties for Interviews
1. **Associative Mapping:** Unlike sequences (lists/tuples) indexed by contiguous integers, dictionaries use a hash table to map keys directly to values.
2. **Ordered (Python 3.7+):** Dictionaries guarantee that elements maintain their **insertion order**. In Python 3.6, this was an implementation detail of CPython; since Python 3.7, it is a formal language specification.
3. **Mutable Container:** The dictionary itself can be modified in-place (adding, updating, or deleting key-value pairs).
4. **Key Constraints:**
   - **Uniqueness:** A key can exist only once in a dictionary. Duplicate keys in literal definitions will overwrite preceding ones.
   - **Hashability:** Keys must be **hashable** (must implement stable `__hash__()` and `__eq__()` methods). This typically restricts keys to immutable types (e.g., `int`, `str`, `tuple`, `frozenset`).
5. **Heterogeneous:** Both keys and values can belong to mixed data types (as long as keys satisfy the hashability rule).

### 🛠️ Initializing Dictionaries
```python
# 1. Direct Literal Initialization
d1 = {"a": 1, "b": 2}

# 2. dict() Constructor with keyword arguments
d2 = dict(a=1, b=2)

# 3. dict() Constructor with an iterable of key-value pairs
d3 = dict([("a", 1), ("b", 2)])

# 4. Dictionary Comprehension (Dynamic Creation)
d4 = {x: x**2 for x in range(3)}
```

In [ ]:
# Demonstrating creation methods and type checks
d_literal = {"name": "Vinod", "role": "Engineer"}
d_constructor = dict(name="Vinod", role="Engineer")
d_iterable = dict([("name", "Vinod"), ("role", "Engineer")])
d_comp = {k: v for k, v in [("name", "Vinod"), ("role", "Engineer")]}

print(f"Literal matches Constructor: {d_literal == d_constructor}")
print(f"Iterable matches Comprehension: {d_iterable == d_comp}")
print(f"Class Type: {type(d_literal)}")

## 🧠 2. Under the Hood: CPython Internals & Hashing Mechanics

To pass senior and system-design level interviews, you must understand CPython's hash table architecture.

### 🔍 A. The Compact Dictionary Layout (PEP 468)
In Python 3.6+, CPython transitioned from a sparse hash table to a **compact dictionary layout**. This changed the memory footprint and enabled insertion ordering.

#### 1. Pre-Python 3.6 Layout (Sparse Hash Table)
The dictionary was a single large, sparse array where each index held a 24-byte entry:
```c
struct PyDictEntry {
    Py_ssize_t me_hash; // Cached hash code
    PyObject *me_key;   // Pointer to key
    PyObject *me_value; // Pointer to value
};
```
To minimize collisions, this array was kept at least $1/3$ empty. For an 8-slot dictionary with 3 items, CPython allocated $8 \times 24 = 192$ bytes. Empty slots sat idle containing null pointers.

#### 2. Modern Compact Layout (Dual-Array Design)
Modern dictionaries separate the sparse hash indices from the dense array of active entries:
1. **`dk_indices` (Sparse Index Array):** A sparse array of integers containing indices mapping to the entries array.
2. **`dk_entries` (Dense Entry Array):** A packed sequential array containing only the actual active entries `[hash, key, value]`.

```
Example: dict = {"z": 100, "a": 200}
If hash("z") maps to index 0, and hash("a") maps to index 5:

dk_indices (Sparse Array of small integers):
[  0,  -1,  -1,  -1,  -1,   1,  -1,  -1  ]
  |                         |
  v                         v
dk_entries[0]             dk_entries[1]

dk_entries (Dense, packed array containing actual elements in insertion order):
[0] -> (hash("z"), pointer_to_z, pointer_to_100)
[1] -> (hash("a"), pointer_to_a, pointer_to_200)
```
- **Memory Optimization:** Since the empty spaces are stored in `dk_indices` as small integers (1 byte each for tables $<128$ slots) rather than full 24-byte C structs, memory consumption is reduced by **20% to 25%**.
- **Insertion Order:** Since `dk_entries` is appended to sequentially, iterating over the dictionary is simply a walk through the dense array, which naturally maintains insertion order.

---

### 🔑 B. Hash Lookup and Equality
When looking up `d[key]`, Python executes the following sequence:
1. Calls `hash(key)` to retrieve the hash value.
2. Computes the target index using a bitwise mask: `index = hash_value & (table_size - 1)`.
3. Checks `dk_indices[index]`:
   - If the index is `-1` (empty slot), a `KeyError` is raised immediately.
   - If the index is $\ge 0$, Python retrieves the entry from `dk_entries[dk_indices[index]]`.
4. Checks identity/equality: It compares `stored_hash == key_hash`. If true, it performs identity check `stored_key is key` or equality check `stored_key == key`.
   - If they match, the value is returned.
   - If they mismatch, a **collision** has occurred.

---

### ⚡ C. Collision Resolution: Pseudo-random Probing
Unlike Java's `HashMap` which uses **Chaining** (linked lists / red-black trees in buckets), Python uses **Open Addressing** with a **Pseudo-random Probing** algorithm to locate the next slot.

The recurrence formula for probing is:
$$\text{next\_index} = (5 \times \text{current\_index} + 1 + \text{perturb}) \pmod{\text{size}}$$
- **Perturb Initial Value:** `perturb` is initialized to the full hash code of the key.
- **Probing Iteration:** In each probing step, `perturb` is shifted right by 5 bits (`perturb >>= 5`). This introduces the higher-order bits of the hash code to break local patterns.
- **Guarantee:** The multiplier of `5` (coprime to power-of-two table sizes) combined with linear congruential generators guarantees that every slot in the hash table will eventually be visited (full period).

---

### 🗑️ D. Deletions and "Tombstones" (`DKIX_DUMMY`)
In open addressing, simply clearing a deleted entry to `-1` breaks existing probing chains.
For example, if key $A$ and key $B$ collide, and $B$ is placed after $A$ along a probe chain, deleting $A$ and clearing its slot would cause subsequent searches for $B$ to fail prematurely at $A$'s empty slot.

- **Solution:** When a key is deleted, its slot in `dk_indices` is marked with a special sentinel integer: `DKIX_DUMMY` (represented as `-2` in CPython).
- **Probing Behavior:**
  - During **lookup**, dummy slots are skipped, allowing the probe chain to continue.
  - During **insertion**, dummy slots can be reclaimed and overwritten by new entries to keep the table compact.

---

### 📈 E. Resizing and Rehashing
- **Load Factor:** CPython maintains a maximum load factor threshold of **$2/3$** ($\approx 66\%$). If $2/3$ or more of the table slots are filled (including dummy slots), resizing is triggered.
- **Sizing Rule:**
  - If the dictionary is small ($< 50,000$ entries), the new table size is **doubled ($2\times$)**.
  - If the dictionary is large, the new table size is **quadrupled ($4\times$)**.
  - Size is always constrained to a power of 2.
- **Rehashing:** A brand-new sparse index array is allocated. Python iterates through `dk_entries`, copies only active keys (discarding all `DKIX_DUMMY` slots), recalculates their indexes using the new size mask, and fills the new sparse array. This shortens lookup probing lengths back to minimum.

In [ ]:
import sys

# Let's inspect the memory allocation jumps in a Python dictionary during growth
d = {}
print(f"Empty Dictionary memory footprint: {sys.getsizeof(d)} bytes")

prev_size = sys.getsizeof(d)
for i in range(100):
    d[i] = i
    curr_size = sys.getsizeof(d)
    if curr_size != prev_size:
        print(f"Length: {len(d):2d} | Dict Size: {curr_size} bytes (Reallocation triggered!)")
        prev_size = curr_size

## ⏳ 3. Time and Space Complexities

Understanding the bounds of dictionary operations is vital for algorithmic interviews.

### Big-O Time Complexity

| Operation | Average Case | Worst Case | Mathematical Reason / Explanation |
| :--- | :--- | :--- | :--- |
| **Get / Lookup (`d[key]`)** | $O(1)$ | $O(N)$ | Average: Hash mapping. Worst: All keys collide, causing a complete traversal of all slots. |
| **Insert / Update (`d[key] = val`)** | $O(1)$ | $O(N)$ | Amortized $O(1)$. Worst: Triggering a table resize and copying/rehashing all $N$ keys. |
| **Delete (`del d[key]`)** | $O(1)$ | $O(N)$ | Average: Look up index and overwrite with `-2`. Worst: Probing through long collision chains. |
| **Containment (`key in d`)** | $O(1)$ | $O(N)$ | Performs a lookup sequence; returns `True` or `False`. |
| **Iteration (`for k in d`)** | $O(N)$ | $O(N)$ | Walks directly through the dense `dk_entries` array from index $0$ to $\text{len}(d) - 1$. |
| **Length (`len(d)`)** | $O(1)$ | $O(1)$ | Direct reading of the size value directly from the header of the C structure. |

### Space Complexity
- **Overall Space Complexity:** $O(N)$ where $N$ is the number of keys.
- **Memory Overhead:** A dictionary occupies significantly more space than a list of tuples containing the same data. This is due to the sparse index array (`dk_indices`), pre-allocated slots for growth, hash code caches, and object headers.

## 🛠️ 4. The Complete Dictionary API (All 11 Built-In Methods)

Python dictionaries have exactly **11 built-in methods**. Here is a systematic breakdown of each method with dedicated code examples.

---

### Category A: Reading and Retrieval

#### 1. `d.get(key, default=None)`
- **Behavior:** Returns the value associated with `key` if it exists. If not, returns the user-specified `default`.
- **Key Interview Point:** Does *not* raise a `KeyError` if the key is missing. Does *not* insert the missing key into the dictionary.

In [ ]:
d = {"name": "Vinod", "role": "Engineer"}

# 1. Retrieve existing key
print("Existing Key lookup:", d.get("name"))

# 2. Retrieve missing key (returns default None)
print("Missing Key lookup:", d.get("email"))

# 3. Retrieve missing key with a custom default
print("Missing Key with custom default:", d.get("email", "no-email@domain.com"))

#### 2. `d.setdefault(key, default=None)`
- **Behavior:** If `key` is in the dictionary, returns its value. If not, inserts `key` with a value of `default` and returns `default`.
- **Key Interview Point:** Unlike `.get()`, this method **mutates the dictionary in-place** when a key is missing.

In [ ]:
d = {"name": "Vinod"}

# 1. Key already exists: returns value, dict unchanged
val1 = d.setdefault("name", "Vikas")
print(f"Key exists -> Returned: {val1} | Dict: {d}")

# 2. Key is missing: inserts key and default value, returns default
val2 = d.setdefault("role", "Engineer")
print(f"Key missing -> Returned: {val2} | Dict: {d}")

#### 3. `d.keys()`
- **Behavior:** Returns a dynamic view object (`dict_keys`) containing all dictionary keys.
- **Key Interview Point:** Reflects changes in real-time. Supports mathematical set-like operations (union `|`, intersection `&`, difference `-`).

In [ ]:
d = {"a": 1, "b": 2}
keys_view = d.keys()
print("Initial keys view:", keys_view)

# Real-time change demonstration
d["c"] = 3
print("Updated keys view (reflects 'c' dynamically):", keys_view)

# Set operation lookup
other_set = {"b", "c", "d"}
print("Keys intersection with other set (&):", keys_view & other_set)

#### 4. `d.values()`
- **Behavior:** Returns a dynamic view object (`dict_values`) containing all dictionary values.
- **Key Interview Point:** Reflects changes in real-time. Unlike keys, the values view **does not support set operations** because values are not guaranteed to be unique.

In [ ]:
d = {"a": 1, "b": 2}
values_view = d.values()
print("Initial values view:", values_view)

# Modify in-place
d["a"] = 99
print("Updated values view (reflects 99 dynamically):", values_view)

#### 5. `d.items()`
- **Behavior:** Returns a dynamic view object (`dict_items`) containing `(key, value)` tuples.
- **Key Interview Point:** Reflects changes in real-time. Supports set operations since key-value pairs are unique.

In [ ]:
d = {"a": 1, "b": 2}
items_view = d.items()
print("Initial items view:", items_view)

# Modify in-place
d["c"] = 3
print("Updated items view:", items_view)

# Set intersection on items
compare_set = {("b", 2), ("z", 9)}
print("Common items:", items_view & compare_set)

---

### Category B: Modification and Creation

#### 6. `d.update(other)`
- **Behavior:** Merges key-value pairs from `other` (another dictionary or an iterable of key-value tuples) into `d` in-place, overwriting overlapping keys.
- **Key Interview Point:** Operates in-place and returns `None`.

In [ ]:
d = {"a": 1, "b": 2}

# Update with a dictionary literal
d.update({"b": 99, "c": 30})
print("After dictionary update:", d)

# Update with an iterable of key-value tuples
d.update([("d", 40), ("e", 50)])
print("After tuple iterable update:", d)

#### 7. `dict.fromkeys(iterable, value=None)`
- **Behavior:** Class method that creates a new dictionary with keys from the `iterable` and all values initialized to `value`.
- **Key Interview Point:** **Dangerous mutable default trap!** If a mutable default value (like `[]` or `{}`) is passed, all keys will reference the *exact same object* in memory.

In [ ]:
# 1. Safe implementation with immutable values (integers)
d1 = dict.fromkeys(["a", "b", "c"], 0)
print("Scalar initialization:", d1)

# 2. Mutable default trap (Danger!)
d2 = dict.fromkeys(["a", "b"], [])
d2["a"].append("shared_item")
print("Trap result (both share same list):", d2)

# 3. Correct work-around using dictionary comprehension
d3 = {k: [] for k in ["a", "b"]}
d3["a"].append("isolated_item")
print("Comp correct result (isolated lists):", d3)

---

### Category C: Removal and Deletion

#### 8. `d.pop(key, default)`
- **Behavior:** Removes `key` and returns its value.
- **Key Interview Point:** If `key` is missing and `default` is provided, returns `default`. If `default` is not provided, raises a `KeyError`.

In [ ]:
d = {"a": 1, "b": 2}

# 1. Pop existing key
val = d.pop("a")
print(f"Popped value: {val} | Remaining Dict: {d}")

# 2. Pop missing key with default
missing_val = d.pop("z", -1)
print(f"Popped missing value with default: {missing_val} | Dict: {d}")

# 3. Pop missing key without default triggers error
try:
    d.pop("z")
except KeyError as e:
    print("pop('z') without default raised KeyError:", e)

#### 9. `d.popitem()`
- **Behavior:** Removes and returns the last inserted `(key, value)` pair as a tuple (LIFO order).
- **Key Interview Point:** Raises a `KeyError` if the dictionary is empty.

In [ ]:
d = {"first": 1, "second": 2, "third": 3}

# Removes last elements in LIFO order
item1 = d.popitem()
print(f"First popped item: {item1} | Dict: {d}")

item2 = d.popitem()
print(f"Second popped item: {item2} | Dict: {d}")

#### 10. `d.clear()`
- **Behavior:** Removes all elements from the dictionary in-place.
- **Key Interview Point:** Keeps the same dictionary object ID in memory; does not create a new empty dictionary.

In [ ]:
d = {"a": 1, "b": 2}
print("Original ID:", id(d))

d.clear()
print("After clear():", d)
print("Same object ID maintained?", id(d))

---

### Category D: Cloning

#### 11. `d.copy()`
- **Behavior:** Returns a **shallow copy** of the dictionary.
- **Key Interview Point:** Creates a new outer container, but nested mutable objects (like lists inside values) are still referenced by pointer to the original memory locations. For nested copies, you must use `copy.deepcopy()`.

In [ ]:
import copy

original = {"a": [1, 2]}
shallow_copied = original.copy()

print(f"New outer container? {shallow_copied is not original}")

# Modify nested mutable array
original["a"].append(3)
print("Shallow copy shares nested lists:", shallow_copied)

# Deep copy isolation demo
deep_copied = copy.deepcopy(original)
original["a"].append(999)
print("Deep copy remains isolated:", deep_copied)

## 🔑 5. Hashability Rules & Custom Key Implementations

An interview question often tests what makes an object hashable and how to implement a custom class as a dictionary key.

### 📐 The Mathematical Definition of Hashability
An object is **hashable** if it satisfies three criteria:
1. It has a hash value that never changes during its lifetime (requires a stable `__hash__()` method).
2. It can be compared to other objects for equality (requires an `__eq__()` method).
3. **The Hash Contract:** If two objects compare equal (`a == b`), they **must** have the exact same hash value (`hash(a) == hash(b)`).

### 🛠️ Creating a Custom Class as a Dictionary Key
By default, custom classes inherit:
- `__hash__()` from `object` (based on their memory address `id()`).
- `__eq__()` which checks identity (`self is other`).
Thus, every instance of a default user class is unique and hashable.

However, if you override `__eq__()` (e.g., to compare values like a data class), Python **automatically sets `__hash__ = None`** to prevent you from violating the Hash Contract. You must explicitly write a matching `__hash__()` method.

In [ ]:
# Implementing a custom class that can safely act as a dictionary key
class Person:
    def __init__(self, name: str, employee_id: int):
        self._name = name
        self._employee_id = employee_id

    @property
    def employee_id(self):
        return self._employee_id

    # 1. Custom Equality (Value-based)
    def __eq__(self, other):
        if not isinstance(other, Person):
            return NotImplemented
        return self._employee_id == other._employee_id

    # 2. Matching Hash function (based on the immutable identifier field)
    def __hash__(self):
        return hash(self._employee_id)

    def __repr__(self):
        return f"Person({self._name}, ID={self._employee_id})"

# Testing the custom key behavior
p1 = Person("Alice", 101)
p2 = Person("Alice Cooper", 101)  # Same ID, evaluates as equal
p3 = Person("Bob", 102)

d = {p1: "Admin Access"}

# p2 should match p1 because they compare equal and hash to the same value
print("p1 == p2:", p1 == p2)
print("p1 hash matches p2 hash:", hash(p1) == hash(p2))
print("Retrieval using p2:", d.get(p2))

## ⚠️ 6. Common Pitfalls, Gotchas, and Edge Cases

Interviewers frequently use these code snippets to screen for practical debugging experience.

---

### Gotcha A: Modifying Dictionary During Iteration
- **Pitfall:** Iterating over a dictionary while adding or deleting keys changes the hash structure and shifts index entries. Python detects this size mutation and raises a `RuntimeError`.
- **Solution:** Create a static list copy of the keys (`list(d.keys())`) to loop over. Modifying values (but not keys) is allowed.

---

### Gotcha B: The `fromkeys()` Shared Mutable Reference Trap
- **Pitfall:** `dict.fromkeys(["a", "b"], [])` initializes all keys to point to the **same list reference in memory**. Modifying the list of one key alters it for all.
- **Solution:** Use a dictionary comprehension: `{key: [] for key in keys}`.

---

### Gotcha C: Key Collisions with Numeric Types
- **Pitfall:** Python treats `True`, `1`, and `1.0` as the exact same key.
- **Reason:**
  - `1 == 1.0 == True` evaluates to `True`.
  - `hash(1) == hash(1.0) == hash(True)` evaluates to `1`.
  - Since hash table lookup checks both `hash(key)` and key equality (`==`), the second inserted key overrides the value of the first, keeping the original key type representation.

---

### Gotcha D: Shallow Copies of Nested Dictionaries
- **Pitfall:** `d.copy()` only replicates the outer container. Nested list or dictionary values are copy-by-reference.
- **Solution:** Use `copy.deepcopy()` to recursively copy all nested objects.

In [ ]:
import copy

# 1. Tricking with fromkeys
bad_dict = dict.fromkeys(["a", "b"], [])
bad_dict["a"].append("shared")
print("1. fromkeys shared value list:", bad_dict)

# 2. Numeric vs Boolean Collision
collision_dict = {1: "Integer One", 1.0: "Float One", True: "Boolean True"}
print("2. Collision dict contents:", collision_dict)
print("   Value lookup for True:", collision_dict[True])

# 3. Shallow vs Deep Copy
original = {"user": {"name": "Alice", "role": "User"}}
shallow = original.copy()
deep = copy.deepcopy(original)

original["user"]["role"] = "Admin"
print("3. Shallow copy nested modification (altered):", shallow["user"]["role"])
print("   Deep copy nested modification (isolated):", deep["user"]["role"])

## 💼 7. Special Dictionary Types (from `collections`)

Standard dictionaries are generic. The `collections` module provides classes that optimize specific search and mapping tasks.

### 1. `collections.defaultdict`
- Automatically initializes missing keys using a user-specified factory function (e.g., `list`, `int`, `set`).
- Eliminates `if key not in dict:` boilerplate check-and-insert blocks.

### 2. `collections.Counter`
- A dictionary subclass designed specifically for counting hashable items.
- Accessing a missing key returns `0` instead of raising a `KeyError`.
- Provides unique utility methods like `.most_common(n)` and arithmetic operators (`+`, `-`, `&`, `|`) for count combining.

### 3. `collections.OrderedDict`
- While standard dictionaries preserve order, `OrderedDict` is still important because:
  - Dictionary equality checks are order-sensitive (unlike standard dicts).
  - Provides control methods: `.move_to_end(key, last=True)` and `.popitem(last=True)` (supports FIFO and LIFO popping).

### 4. `collections.ChainMap`
- Groups multiple dictionaries into a single logical mapping view. Lookups search through dictionaries sequentially from first to last.
- **Complexity Advantage:** Saves overhead by avoiding dictionary merging (`|`) which requires copying keys. Space complexity is $O(1)$ memory mapping.

### 5. `collections.UserDict`
- A wrapper around dictionary objects. It serves as a safer base class for creating custom dictionary subclasses than subclassing `dict` directly (which has optimization overrides that bypass user-defined overrides).

In [ ]:
from collections import defaultdict, Counter, ChainMap

# 1. defaultdict grouping example
fruits = [("apple", 1), ("banana", 2), ("apple", 3)]
d_grouped = defaultdict(list)
for fruit, qty in fruits:
    d_grouped[fruit].append(qty)
print("defaultdict grouped:", dict(d_grouped))

# 2. Counter count combining
c1 = Counter(a=3, b=1)
c2 = Counter(a=1, b=2, c=5)
print("Counter sum (+):", c1 + c2)
print("Counter intersection (&):", c1 & c2)

# 3. ChainMap lookup order
defaults = {"theme": "light", "show_sidebar": True}
user_settings = {"theme": "dark"}
config = ChainMap(user_settings, defaults)
print("ChainMap lookup ('theme'):", config["theme"])
print("ChainMap lookup ('show_sidebar'):", config["show_sidebar"])

## 🏆 8. Top Coding Interview Patterns & Practical Use Cases

These are high-frequency coding patterns where dictionaries provide optimal performance.

---

### Pattern A: $O(1)$ Complement Hash Map (Two Sum)
- **Concept:** Check if the required mathematical complement `target - value` exists in the hash map. If yes, retrieve its metadata in $O(1)$.
- **Time Complexity:** $O(N)$ runtime vs $O(N^2)$ brute-force lookup.

---

### Pattern B: Prefix Sum / Cumulative Map (Subarray Sum Equals K)
- **Concept:** Store prefix sums and their frequency in a dictionary. Allows checking if a subsegment sums to `k` in $O(1)$ lookup time.
- **Time Complexity:** $O(N)$ runtime.

---

### Pattern C: LRU Cache (Least Recently Used Cache)
- **Concept:** An LRU cache requires $O(1)$ reads (`get`) and $O(1)$ writes (`put`). When capacity is reached, it must evict the least recently used element.
- **Architecture:** We combine a Doubly Linked List (for tracking recency order in $O(1)$) and a Hash Map (for $O(1)$ lookup of nodes). In Python, we implement this natively using `collections.OrderedDict`.

In [ ]:
from collections import OrderedDict

# Pattern A: Two Sum implementation
def two_sum(nums, target):
    seen = {}
    for index, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], index]
        seen[num] = index
    return []

print("Pattern A (Two Sum):", two_sum([3, 2, 4], 6))

# Pattern B: Subarray Sum Equals K implementation
def subarray_sum(nums, k):
    count = 0
    prefix_sums = {0: 1}
    curr_sum = 0
    for num in nums:
        curr_sum += num
        if (curr_sum - k) in prefix_sums:
            count += prefix_sums[curr_sum - k]
        prefix_sums[curr_sum] = prefix_sums.get(curr_sum, 0) + 1
    return count

print("Pattern B (Subarray Sum Count):", subarray_sum([1, 1, 1], 2))

# Pattern C: LRU Cache implementation
class LRUCache:
    def __init__(self, capacity: int):
        self.cache = OrderedDict()
        self.capacity = capacity

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key: int, value: int) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)

lru = LRUCache(2)
lru.put(1, 10)
lru.put(2, 20)
lru.get(1)
lru.put(3, 30)
print(f"Pattern C (LRU Cache): Key 2 exists? {lru.get(2) != -1} | Key 1 exists? {lru.get(1) != -1}")

## 🏁 Summary & Interview Checklist

Ensure you are solid on these checkpoints before your interview:

- [ ] **Dual Array Layout:** Understand the compact dictionary representation: the separation of the sparse `dk_indices` index array and dense `dk_entries` entry array.
- [ ] **Collision Probing:** Explain that Python does not use linked-list buckets; it resolves collisions via **Open Addressing** with **Pseudo-random Probing** using `(5 * i + 1 + perturb) % size`.
- [ ] **Deletion Tombstones:** Know that CPython uses `-2` (`DKIX_DUMMY`) placeholders to keep probing chains unbroken on deletions.
- [ ] **Method Return Behavior:** Understand that `.pop()` raises a `KeyError` unless a default value is supplied, and `.setdefault()` modifies the dictionary in-place.
- [ ] **View Object Efficiency:** Dynamic view objects (`keys()`, `values()`, `items()`) do not allocate copy-lists and dynamically link to modifications in real-time.
- [ ] **Hash Contract:** Know that a custom class key requires overriding both `__eq__()` and `__hash__()`, maintaining the invariant that if `a == b`, then `hash(a) == hash(b)`.
- [ ] **Pitfalls Checklist:**
  - Avoid mutating keys during iteration.
  - Avoid sharing mutable default list objects in `dict.fromkeys()`.
  - Watch for numeric key collisions like `1`, `1.0`, and `True` merging into the same slot.
- [ ] **Collections Variants:** Know when to use `Counter` (counts), `defaultdict` (missing-key defaults), and `ChainMap` (zero-copy dict stack lookups).